In [1]:
import os
os.environ["TRANSFORMERS-VERBOSITY"]="error"
import sqlite3
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings



d:\ksn-II\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
"""
Ingests resolved tickets from data/tickets.db into the 'tickets Chroma collection.
Run once (or after adding new tickets): python ingestion_tickets.ipynb
"""

"\nIngests resolved tickets from data/tickets.db into the 'tickets Chroma collection.\nRun once (or after adding new tickets): python ingest_tickets.py\n"

In [13]:

CHROMA_DIR="Chroma_store"
COLLECTION="tickets"
DB_PATH=os.path.join("data","tickets.db")
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

In [14]:
def load_ticket_documents(db_path:str)->list[Document]:
    conn=sqlite3.connect(db_path)
    conn.row_factory=sqlite3.Row
    rows=conn.execute(
        "SELECT * FROM tickets WHERE status = 'resolved'"
    ).fetchall()
    conn.close()

    docs=[]
    for row in rows:
        #description+resolution
        content=(
            f"Issue : {row['issue_type']}\n"
            f"Description:{row['description']}\n"
            f"Resolution : {row['resolution']}"
        )
        
        docs.append(
            Document(
                page_content=content,
                metadata={
                    "source": "tickets",
                    "ticket_id": row["ticket_id"],
                    "category": row["category"],
                    "status": row["status"],
                },
            )
        )

    return docs

In [15]:
import os

print(DB_PATH)
print(os.path.exists(DB_PATH))

data\tickets.db
True


In [16]:
def main():
    print("Loading tickets documents form SQLite...")
    docs= load_ticket_documents(DB_PATH)
    print(f"{len(docs)} resolved tickets loaded.")

    print("Initialising embedding model...")
    embedding=HuggingFaceEmbeddings(model_name=EMBED_MODEL)

    print(f"Embedding and storing in Chroma collection '{COLLECTION}' ...")

    vectore_store=Chroma.from_documents(
        documents=docs,
        embedding=embedding,
        collection_name=COLLECTION,
        persist_directory=CHROMA_DIR,
    )
    print (f" Done. {vectore_store._collection.count()} vectors stored.")

In [17]:
docs = load_ticket_documents(DB_PATH)

print(docs)
print(type(docs))

[Document(metadata={'source': 'tickets', 'ticket_id': 'TK-001', 'category': 'connectivity', 'status': 'resolved'}, page_content='Issue : No internet access\nDescription:Customer reports complete loss of mobile internet after switching from 3G to 4G mode.\nResolution : Guided customer to toggle airplane mode and reset APN settings. Internet restored after APN was reset to default (internet.telecom.example).'), Document(metadata={'source': 'tickets', 'ticket_id': 'TK-002', 'category': 'connectivity', 'status': 'resolved'}, page_content='Issue : Intermittent signal drops\nDescription:Customer in a suburban area experiences signal dropping to zero several times per day, affecting both calls and data.\nResolution : Network team identified a faulty sector antenna on the nearest tower. Temporary fix applied remotely; full hardware replacement scheduled within 48 hours. Customer offered 500 MB data compensation.'), Document(metadata={'source': 'tickets', 'ticket_id': 'TK-003', 'category': 'dat

In [18]:
print(main())

Loading tickets documents form SQLite...
19 resolved tickets loaded.
Initialising embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2151.20it/s]


Embedding and storing in Chroma collection 'tickets' ...
 Done. 19 vectors stored.
None
